# Greek Electronic Auctions Scraper — eauction.gr

This notebook scrapes property auction data from [eauction.gr](https://www.eauction.gr), the official Greek electronic auction platform.

**Pipeline overview**

1. Scrape auction listing pages to collect auction URLs and regions
2. Combine and deduplicate listing CSVs; extract clean URLs with regex
3. Visit each auction detail page and extract structured fields (dates, bids, debtors, PDFs)
4. Merge all data into a master dataset

**Key techniques:** Selenium automation, BeautifulSoup parsing, regex extraction, pandas data wrangling

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

import pandas as pd
import glob
import re
import time
import os
from urllib.parse import urljoin
from urllib.request import urlopen
import urllib
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.support.ui import WebDriverWait as wait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common import exceptions
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import undetected_chromedriver as uc

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

---
## Part 1 — Scrape Auction Listings

Iterates through listing pages to collect auction entry URLs and region labels.
Each page returns up to 20 results. Results are saved to CSV for later processing.

In [ ]:
driver = uc.Chrome(version_main=147)
driver.get("https://www.eauction.gr")
time.sleep(5)  # let the PoW challenge complete on its own

In [ ]:
#SCRAPE AND SAVE THE RESULTS INTO A DICTIONARY. THEN APPEND INTO A LIST

entries_all = []
    
for page in range(0,10):
    driver.get("https://www.eauction.gr/en/Home/HlektronikoiPleistiriasmoi?conductFrom=01/07/2026&sortAsc=true&sortId=1&conductedSubTypeId=1&page=" + str(page))
    time.sleep(5)
    time.sleep(1/1)
    
    doc = BeautifulSoup(driver.page_source, 'html.parser')
    each_entry = doc.find_all(class_="AList-BoxContainer")[0:20]
    for entry in each_entry:
        all_entries_dict ={}
        
        entry_object1 = entry.find('div', class_="AList-BoxMainCell4")
        entry_object2 = entry_object1.find('div', class_="AList-BoxTextBlueBold")
        entry_location = entry_object1.find('div', class_="AList-BoxTextBlue")
        
        entry_more_info = entry.find(class_="AList-BoxFooterMore")
                
        all_entries_dict['Location'] = entry_location
        
        all_entries_dict['More_Info'] = entry_more_info
        
        entries_all.append(all_entries_dict)
        print(all_entries_dict)
        print('-----')
        #time.sleep(2)
#     try:
#         next_button = driver.find_element_by_id('pagerNext')
#         driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#         next_button.click()
#         time.sleep(3)
#     except:
#         next_button.click()
#         time.sleep(3)

In [ ]:
df = pd.DataFrame(entries_all)
df = df.drop_duplicates(subset=['More_Info'], keep='first')
df

In [ ]:
df.to_csv("data/links/01_Links_Regions_2026_07_Jan_p1_10.csv", index=False)

---
## Part 2 — Combine Listing CSVs & Clean URLs

Loads all listing CSVs from `data/links/`, combines and deduplicates them, then uses
regex to extract the clean URL from the raw HTML anchor stored in `More_Info`.

In [ ]:
# Load all listing CSVs
files = glob.glob("data/links/*.csv")
print("Files found:", files)

first_scrape_dfs = [pd.read_csv(f) for f in files]

for f, dataframe in zip(files, first_scrape_dfs):
    dataframe['filename'] = f

combined_df = pd.concat(first_scrape_dfs, ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=['More_Info'], keep='first')
combined_df.shape

In [ ]:
combined_df.to_csv("01_Links_Regions_2026_07.csv", index=False)

In [ ]:
# Extract clean URL from raw HTML anchor with regex
df2 = pd.read_csv("01_Links_Regions_2026_07.csv")
df2["More_Info"] = df2["More_Info"].str.extract(r'href="([^"]+)"')
df2.to_csv("01_1st_scrape_CLEAN_URL.csv", index=False)
df2

---
## Part 3 — Scrape Auction Detail Pages

Visits each auction URL and extracts structured fields:
- Auction state, dates, starting bid, total debt
- Object type, real estate type, region, municipality  
- Hastener and auction employee details
- Debtor names and VAT numbers (supports multiple debtors)
- Attached PDF documents (type, name, absolute link)

Includes a stop/resume mechanism: if the page structure is missing (captcha, block),
the loop prints the current index and URL, then breaks so you can resume from that point.

In [ ]:
df = pd.read_csv("01_1st_scrape_CLEAN_URL.csv")
df

In [ ]:
driver = uc.Chrome(version_main=147)
driver.get("https://www.eauction.gr")
time.sleep(5)  # let the PoW challenge complete on its own

In [ ]:
BASE_URL = "https://www.eauction.gr"


def normalize_text(s):
    if s is None:
        return None
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"(?<=\d)\s+(?=\d)", "", s)
    s = re.sub(r"(\d{2}/\d{2}/\d{4})(\d{1,2}:\d{2})", r"\1 \2", s)
    s = s.replace("€", " €")
    s = re.sub(r"\s+", " ", s).strip()
    return s or None


def clean_text(el):
    if not el:
        return None
    return normalize_text(el.get_text("", strip=True))


def norm_label(s):
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip().rstrip(":")
    for ch in ("`", "\u2019", "\u00b4"):
        s = s.replace(ch, "'")
    return s


def is_hidden_placeholder(tag):
    cls = " ".join(tag.get("class", []))
    style = (tag.get("style") or "").replace(" ", "").lower()
    return "ResetADetailsinputDateOn" in cls or "text-indent:-9999px" in style


def label_values(doc, label_text):
    want = norm_label(label_text)
    out = []
    for lab in doc.find_all("label"):
        if norm_label(lab.get_text("", strip=True) or "") != want:
            continue
        container = lab.find_parent("div", class_=lambda c: c and "AuctionDetailsDiv" in c)
        if not container:
            continue
        for cand in container.find_all(["label", "div"], class_=lambda c: c and "ADetailsinput" in c):
            if is_hidden_placeholder(cand):
                continue
            v = clean_text(cand)
            if v:
                out.append(v)
    return out


def label_value_first(doc, label_text):
    vals = label_values(doc, label_text)
    return vals[0] if vals else None


def parse_debtors(doc):
    vals = label_values(doc, "Debtors' Names and Surnames")
    if vals:
        return [v for v in vals if v.strip()]
    v = label_value_first(doc, "Debtor`s Name and Surname")
    return [v] if v else []


def parse_debtor_vats(doc):
    vals = label_values(doc, "Debtors' Vat Numbers")
    if vals:
        return [v for v in vals if v.strip()]
    v = label_value_first(doc, "Debtor`s VAT Number")
    return [v] if v else []


def parse_pdfs(doc):
    pdfs = []
    for item in doc.select(".AuctionDetailsPDFCnt .AuctionDetailsPDFItem"):
        pdf_type = clean_text(item.select_one(".AuctionDetailsPDFlbl"))
        a = item.select_one(".AuctionDetailsPDFtext a[href]")
        if not a:
            continue
        href = a.get("href")
        pdfs.append({
            "type": pdf_type,
            "name": clean_text(a),
            "link": urljoin(BASE_URL, href) if href else None
        })
    return pdfs


def set_if(d, k, v):
    if v is not None and (not isinstance(v, str) or v.strip()):
        d[k] = v


def wait_for_page(driver, url, max_attempts=3, wait_seconds=6):
    """Navigate to url and wait for the PoW challenge to resolve. Returns parsed doc or None."""
    driver.get(url)
    for attempt in range(max_attempts):
        time.sleep(wait_seconds)
        doc = BeautifulSoup(driver.page_source, "html.parser")
        if doc.find("div", class_="AuctionDetailsSection1"):
            return doc
        print(f"  PoW challenge on attempt {attempt + 1}/{max_attempts}, waiting...")
    return None


# --- Main loop ---
entries_all2 = []

start_idx = 0   # resume from here (inclusive)
end_idx   = 10  # exclusive — set to len(df) to process all
end_idx = min(end_idx, len(df))

for i in range(start_idx, end_idx):
    item = df["More_Info"].iloc[i]
    if not item or len(str(item).strip()) <= 1:
        continue

    url = str(item).strip()
    time.sleep(6)

    doc = wait_for_page(driver, url)

    if doc is None:
        print(f"Skipping index {i} after failed attempts: {url}")
        continue

    row = {"More_Info": url}

    set_if(row, "Auction_State",        clean_text(doc.select_one(".StateBox .StateValue")))
    set_if(row, "Auction_Date",         label_value_first(doc, "Date of Conduction"))
    set_if(row, "Posting_Date",         label_value_first(doc, "Date of Posting"))
    set_if(row, "First_Bid",            label_value_first(doc, "Starting Bid"))
    set_if(row, "Total_Debt",           label_value_first(doc, "Total Debt"))
    set_if(row, "Object_Type",          label_value_first(doc, "Object to be auctioned"))
    set_if(row, "Real_Estate_Type",     label_value_first(doc, "Real Estate Type"))
    set_if(row, "Region",               label_value_first(doc, "Region"))
    set_if(row, "Municipality",         label_value_first(doc, "Municipality"))
    set_if(row, "Publication_Date",     label_value_first(doc, "Publication Date"))
    set_if(row, "Unique_Code",          label_value_first(doc, "Unique Code"))
    set_if(row, "Duration_Extensions",  label_value_first(doc, "Duration - Extensions"))
    set_if(row, "Hastener",             label_value_first(doc, "Hastener"))
    set_if(row, "Hastener_VAT",         label_value_first(doc, "Hastener`s VAT number"))
    set_if(row, "Auction_Employee",     label_value_first(doc, "Auction Employee"))
    set_if(row, "Employee_Address",     label_value_first(doc, "Address"))
    set_if(row, "Employee_Phone",       label_value_first(doc, "Phone"))
    set_if(row, "Employee_Email",       label_value_first(doc, "E-mail"))
    set_if(row, "Remarks",              label_value_first(doc, "Remarks"))

    debtor_names = parse_debtors(doc)
    debtor_vats  = parse_debtor_vats(doc)
    n = max(len(debtor_names), len(debtor_vats))
    for j in range(n):
        if j < len(debtor_names): set_if(row, f"Debtor_{j+1}_Name", debtor_names[j])
        if j < len(debtor_vats):  set_if(row, f"Debtor_{j+1}_VAT",  debtor_vats[j])
    row["Debtor_Count"] = n

    pdfs = parse_pdfs(doc)
    for k, p in enumerate(pdfs, start=1):
        set_if(row, f"PDF_{k}_Type", p.get("type"))
        set_if(row, f"PDF_{k}_Name", p.get("name"))
        set_if(row, f"PDF_{k}_Link", p.get("link"))
    row["PDF_Count"] = len(pdfs)

    entries_all2.append(row)
    print(row)
    print("-----")

In [ ]:
# Merge detail rows back onto the listings dataframe
an_df = pd.DataFrame(entries_all2)
an_df = df.merge(an_df, on="More_Info")
print(an_df.shape)
an_df.to_csv("data/details/03_All_Details_2026_JUL_batch_01.csv", index=False)
an_df

---
## Part 4 — Combine All Detail Batches & Export Master Dataset

Loads all detail batch CSVs, concatenates and deduplicates them, and saves the final master dataset.

In [ ]:
files = sorted(glob.glob('data/details/*.csv'))
print("Detail files found:", files)

auctions_dfs = [pd.read_csv(f) for f in files]

for f, dataframe in zip(files, auctions_dfs):
    dataframe['filename'] = f

combined_df = pd.concat(auctions_dfs, ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=['More_Info'], keep='first')
print("Master dataset shape:", combined_df.shape)

In [ ]:
combined_df.to_csv("000_MASTER_DATASET_JUL_2026.csv", index=False)
combined_df.to_csv("00_2026_COMBINED_FOR_OPENREFINE.csv", index=False)

In [ ]:
df_master = pd.read_csv("000_MASTER_DATASET_JUL_2026.csv")
print(df_master.shape)
df_master.dtypes